# Stopwords and Stemming
We start from the clean text of version 2.
We decide everything using the train data only.
Then we use the same choices on the test data.

## 1. Load the data

In [1]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
train = pd.read_csv('../data/train_clean_v2.csv')
test = pd.read_csv('../data/test_clean_v2.csv')
print(train.shape, test.shape)
train[['label', 'clean_text']].head()

(4102, 7) (1026, 7)


,label,clean_text
0,ham,hey we can go jazz power yoga hip hop kb and y...
1,spam,please call amanda with regard to renewing or ...
2,ham,whats nannys address
3,ham,and maybe some pressies
4,ham,sorry was in the bathroom sup


## 2. The stopwords list
Stopwords are very common small words, like "the", "to" and "my".
We use the English list from NLTK.
We remove the apostrophes from the list, because our clean text has none.

In [2]:
nltk.download('stopwords')
stop_words = set()
for w in stopwords.words('english'):
    stop_words.add(w.replace("'", ''))
print(len(stop_words))

196


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\iaReemy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 3. Are stopwords the same in ham and spam?
For each stopword, we find the percent of ham messages and spam messages that have it.
ratio = spam percent / ham percent.
We add 0.1 to both, so we never divide by zero.
A big ratio means the word is more in spam. A small ratio means it is more in ham.
We only look at words that are in at least 3% of ham or spam messages.

In [3]:
ham = train['label'] == 'ham'
spam = train['label'] == 'spam'
rows = []
for w in sorted(stop_words):
    has = train['clean_text'].str.contains(r'\b' + w + r'\b')
    rows.append([w, round(100 * has[ham].mean(), 1), round(100 * has[spam].mean(), 1)])
table = pd.DataFrame(rows, columns=['word', 'ham_pct', 'spam_pct'])
table['ratio'] = (table['spam_pct'] + 0.1) / (table['ham_pct'] + 0.1)
common = table[(table['ham_pct'] >= 3) | (table['spam_pct'] >= 3)]
print('Common stopwords:', len(common))
print(common.sort_values('ratio', ascending=False).to_string())

Common stopwords: 61
      word  ham_pct  spam_pct      ratio
182    won      0.0       9.0  91.000000
113    our      1.1      10.8   9.083333
46    from      3.3      17.4   5.147059
111     or      4.9      24.2   4.860000
190   your      7.4      30.7   4.106667
104    now      6.2      25.3   4.031746
110   only      3.0      10.2   3.322581
177    who      1.1       3.6   3.083333
181   with      4.7      13.6   2.854167
19    been      1.8       5.0   2.684211
45     for      9.4      24.4   2.578947
51     has      1.5       4.0   2.562500
152   this      5.1      12.8   2.480769
155     to     25.8      60.9   2.355212
93    more      1.5       3.6   2.312500
108     on      7.5      16.6   2.197368
0        a     18.2      37.1   2.032787
54    have      8.5      16.4   1.918605
116    out      4.3       7.8   1.795455
86    just      6.1      10.6   1.725806
26      by      2.6       4.4   1.666667
15      as      2.5       4.2   1.653846
77      is     11.8      18.2   1.53

## 4. Our own stopwords list
We keep a stopword if one class uses it at least 2 times more than the other class.
These words help to tell ham from spam.
We remove all the other stopwords.

In [4]:
keep = common[(common['ratio'] >= 2) | (common['ratio'] <= 0.5)]['word'].tolist()
remove = stop_words - set(keep)
print('Keep:', len(keep))
print(keep)
print('Remove:', len(remove))

Keep: 39
['a', 'about', 'at', 'been', 'but', 'do', 'dont', 'for', 'from', 'has', 'he', 'how', 'i', 'ill', 'im', 'it', 'its', 'me', 'more', 'my', 'not', 'now', 'on', 'only', 'or', 'our', 'so', 'that', 'then', 'this', 'to', 'up', 'was', 'what', 'when', 'who', 'with', 'won', 'your']
Remove: 157


## 5. Remove stopwords
We try two ways and compare them:
- all_stop: remove all 196 NLTK stopwords.
- stop_text: remove only the 157 words in our own list.

In [5]:
def remove_stop(text, words_to_remove):
    kept = []
    for w in text.split():
        if w not in words_to_remove:
            kept.append(w)
    return ' '.join(kept)

train['all_stop'] = train['clean_text'].apply(remove_stop, args=(stop_words,))
train['stop_text'] = train['clean_text'].apply(remove_stop, args=(remove,))
for col in ['clean_text', 'all_stop', 'stop_text']:
    print(col, '| words:', train[col].str.split().str.len().sum(),
          '| different words:', len(set(' '.join(train[col]).split())),
          '| empty:', (train[col] == '').sum())

clean_text | words: 63880 | different words: 7303 | empty: 0
all_stop | words: 37752 | different words: 7137 | empty: 3
stop_text | words: 51804 | different words: 7176 | empty: 1


## 6. Messages that became empty
Some short messages have only stopwords.
After removing the stopwords, nothing is left.

In [6]:
print(train[train['all_stop'] == '']['text'].tolist())
print(train[train['stop_text'] == '']['text'].tolist())

['Where @', 'Can a not?', 'What you doing?how are you?']
['Where @']


## 7. Our choice
We use our own list (stop_text), because it keeps the words that help.
If a message becomes empty, we keep its clean text, so no message is lost.

In [7]:
train.loc[train['stop_text'] == '', 'stop_text'] = train['clean_text']
train = train.drop(columns='all_stop')
print('Empty:', (train['stop_text'] == '').sum())
for i in [0, 3]:
    print('CLEAN:', train['clean_text'][i])
    print('STOP :', train['stop_text'][i])
    print()

Empty: 0
CLEAN: hey we can go jazz power yoga hip hop kb and yogasana
STOP : hey go jazz power yoga hip hop kb yogasana

CLEAN: and maybe some pressies
STOP : maybe pressies



## 8. Stemming
Stemming cuts the end of a word to get its root.
For example "running" becomes "run".
We use the Porter stemmer.
Here we test it on some words.

In [8]:
ps = PorterStemmer()
for w in ['running', 'messages', 'university', 'universe', 'like', 'likely',
          'use', 'useful', 'go', 'went', 'happyface', 'longnumber']:
    print(w, '->', ps.stem(w))

running -> run
messages -> messag
university -> univers
universe -> univers
like -> like
likely -> like
use -> use
useful -> use
go -> go
went -> went
happyface -> happyfac
longnumber -> longnumb


## 9. Stem our text
We do not stem our special words (like happyface and longnumber),
so they stay the same.
Then we see which words became one root.

In [9]:
special = ['happyface', 'sadface', 'hiddeninfo', 'tandc', 'url',
           'pounds', 'dollars', 'pence', 'longnumber']

def stem_text(text):
    out = []
    for w in text.split():
        if w in special:
            out.append(w)
        else:
            out.append(ps.stem(w))
    return ' '.join(out)

vocab = sorted(set(' '.join(train['stop_text']).split()))
pairs = pd.DataFrame({'word': vocab, 'stem': [stem_text(w) for w in vocab]})
print('Different words before:', len(pairs), '| after stemming:', pairs['stem'].nunique())
groups = pairs.groupby('stem')['word'].apply(list)
print(groups[groups.str.len() >= 5].to_string())

Different words before: 7177 | after stemming: 5967
stem
care        [care, cared, careful, carefully, cares, caring]
complet    [complete, completed, completely, completes, c...
hope       [hope, hopeful, hopefully, hopeing, hopes, hop...
like                    [like, liked, likely, likes, liking]
love                    [love, loved, lovely, loves, loving]
predict    [predict, predicte, predicting, prediction, pr...
remind      [remind, reminded, reminder, reminding, reminds]
use                         [use, used, useful, uses, using]


## 10. Use the same steps on test, then save
We use the same stopwords list (made from train) and the same stemmer on the test data.
We save three versions of the text:
- clean_text: after cleaning
- stop_text: after removing our stopwords
- stem_text: after removing our stopwords and stemming
Later we can test which version works best.

In [10]:
test['stop_text'] = test['clean_text'].apply(remove_stop, args=(remove,))
test.loc[test['stop_text'] == '', 'stop_text'] = test['clean_text']
for df in [train, test]:
    df['stem_text'] = df['stop_text'].apply(stem_text)
train.to_csv('../data/train_prep_v2.csv', index=False)
test.to_csv('../data/test_prep_v2.csv', index=False)
check_train = pd.read_csv('../data/train_prep_v2.csv')
check_test = pd.read_csv('../data/test_prep_v2.csv')
print(check_train.shape, check_test.shape)
print('Empty cells:', check_train.isna().sum().sum(), check_test.isna().sum().sum())
print(list(check_train.columns))

(4102, 9) (1026, 9)
Empty cells: 0 0
['label', 'text', 'length', 'caps', 'excl', 'dots', 'clean_text', 'stop_text', 'stem_text']
